# 1. Device discovery

Learn the difference between configured identities, USB candidates, and a verified controller.


## Before you start

Follow [the notebook setup and safety guide](README.md), then select the
project's Python 3.12 kernel. Run cells from top to bottom. Each notebook
is independent. Default execution uses a fictional config and makes no
hardware calls; a skipped operation is **not a successful hardware test**.

For an authorized live session, set `CONFIG` to `ROOT / 'shutters.local.toml'`,
set `TARGET` to the reviewed exact device name, and only then enable the
relevant gate. Keep the source independently protected. Release any GUI,
CLI or other notebook owner first. Restart the kernel after changing setup.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from pprint import pprint
from thorlabs_shutter_control import ShutterHub, load_config

# Works from the repository root or docs/examples. No device access.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/shutters.example.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project checkout.')
CONFIG = ROOT / 'config/shutters.example.toml'  # Fictional; offline only.
TARGET = 'example'  # Exact configured name; never select the first device.
RUN_HARDWARE = False

configs = load_config(CONFIG)
selected = next((c for c in configs if c.name == TARGET), None)
if selected is None:
    raise ValueError('TARGET must match an exact configured name.')
print('Selected:', selected.name, selected.serial_number, selected.route.kind)


## 1. List configured devices (offline)

Configuration is an address book, not a live discovery result. The supplied
serial and firmware are fictional. Read the actual controller label and
approved firmware record before building local configuration.


In [ ]:
for item in configs:
    pprint({'name': item.name, 'serial': item.serial_number,
            'firmware_allowlist': item.firmware,
            'route': asdict(item.route), 'role': item.role})


## 2. Optional Windows COM-port inventory

This explicit opt-in enumerates local serial-port metadata using the `vcp`
extra. It does not open ports or identify a KSC101. A USB description or
serial may identify a bridge rather than the shutter controller. Do not
connect to every candidate, change drivers, or automatically choose a port.
D2XX devices may have no COM port; use their exact reviewed serial in config.


In [ ]:
RUN_PORT_INVENTORY = False
if RUN_PORT_INVENTORY:
    from serial.tools import list_ports
    candidates = sorted(list_ports.comports(include_links=False),
                        key=lambda p: p.device)
    for port in candidates:
        pprint({'port': port.device, 'description': port.description,
                'usb_serial': port.serial_number, 'vid': port.vid, 'pid': port.pid})
    if not candidates:
        print('No COM candidates. This does not prove a D2XX device is absent.')
else:
    print('Inventory skipped; no ports enumerated.')


## 3. Explicit CLI inventory and next steps

For FTDI metadata use `uv run --locked --extra d2xx shutter-control discover`.
For COM-port metadata use `uv run --locked --extra vcp shutter-control discover --vcp`.
These commands need no config, open no controller and never select a target.
They may list other instruments; neither result verifies a KSC101 identity.
Record exact identities and routes manually, then use notebook 2 to verify
the selected controller. A firmware mismatch is a reason to investigate,
not to broaden the allowlist. KEH3 network scanning and direct Ethernet
control are unavailable; see [the Ethernet assessment](../ethernet-assessment.md).
Do not substitute a guessed TCP address or fall back to another device.
